# CS7640 Phase 2 Colab Runner (From Project Code)

This notebook runs your existing repository scripts (`train.py`, `eval.py`, `report.py`) directly so Colab behavior matches your latest code.

Before running:
1. Upload your full project folder to Google Drive.
2. Ensure your dataset is in the expected split structure (`train/HE`, `train/IHC`, `val/HE`, `val/IHC`, and optional `test/*`).
3. Edit the path variables in the next cell.


In [ ]:
# ===== User settings (edit these) =====
USE_DRIVE = True
PROJECT_ROOT = "/content/drive/MyDrive/CS7640-Project"   # folder containing train.py
DATA_ROOT = "/content/drive/MyDrive/CS7640/data/raw"      # dataset root
OUTPUT_ROOT = "/content/drive/MyDrive/CS7640/outputs"     # where outputs will be saved
FAST_DEBUG = False  # True = short smoke test (2 epochs each)


In [ ]:
# Mount Google Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# Validate project paths and switch working directory
import os
import sys

required_files = [
    "train.py",
    "eval.py",
    "report.py",
    "configs/base.yaml",
    "configs/expr.yaml",
    "requirements.txt",
]

assert os.path.isdir(PROJECT_ROOT), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
for rel in required_files:
    p = os.path.join(PROJECT_ROOT, rel)
    assert os.path.exists(p), f"Missing file: {p}"

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Working directory:", os.getcwd())
print("Data root:", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)


In [ ]:
# Install dependencies
%pip -q install -r requirements.txt


In [ ]:
# Runtime check
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Dataset sanity check
import os

for split in ["train", "val"]:
    for stain in ["HE", "IHC"]:
        p = os.path.join(DATA_ROOT, split, stain)
        n = len(os.listdir(p)) if os.path.isdir(p) else 0
        print(f"{p}: {n} files")

for stain in ["HE", "IHC"]:
    p = os.path.join(DATA_ROOT, "test", stain)
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"{p}: {n} files")


## Build Colab-Specific Configs

This cell reads your existing `configs/base.yaml` and `configs/expr.yaml`, updates dataset/output paths for Colab, and writes:
- `configs/base.colab.yaml`
- `configs/expr.colab.yaml`


In [ ]:
import copy
import yaml
import os


def load_yaml(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def save_yaml(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(obj, f, sort_keys=False)


base_cfg = load_yaml("configs/base.yaml")
expr_cfg = load_yaml("configs/expr.yaml")

base_cfg = copy.deepcopy(base_cfg)
expr_cfg = copy.deepcopy(expr_cfg)

# Path overrides for Colab
base_cfg["data"]["root_dir"] = DATA_ROOT
expr_cfg["data"]["root_dir"] = DATA_ROOT
base_cfg["training"]["checkpoint_dir"] = os.path.join(OUTPUT_ROOT, "base")
expr_cfg["training"]["checkpoint_dir"] = os.path.join(OUTPUT_ROOT, "expr")

# Colab-friendly dataloader workers
base_cfg["data"]["num_workers"] = int(min(2, base_cfg["data"].get("num_workers", 2)))
expr_cfg["data"]["num_workers"] = int(min(2, expr_cfg["data"].get("num_workers", 2)))

# Optional fast debug mode
if FAST_DEBUG:
    for cfg in (base_cfg, expr_cfg):
        cfg["training"]["epochs"] = 2
        cfg["training"]["val_every"] = 1
        cfg["training"]["save_every"] = 1

base_colab = "configs/base.colab.yaml"
expr_colab = "configs/expr.colab.yaml"
save_yaml(base_cfg, base_colab)
save_yaml(expr_cfg, expr_colab)

print("Wrote:", base_colab)
print("Wrote:", expr_colab)
print("FAST_DEBUG:", FAST_DEBUG)


## Train Models
Run these cells in order. They can take a long time on full settings.


In [ ]:
# Step 1: Train M_base
import subprocess

subprocess.run(["python", "train.py", "--config", "configs/base.colab.yaml"], check=True)


In [ ]:
# Step 2: Train M_expr
import subprocess

subprocess.run(["python", "train.py", "--config", "configs/expr.colab.yaml"], check=True)


In [ ]:
# Helpers for latest run/checkpoint/metrics
import os
import re
from pathlib import Path


def run_index(path_obj):
    m = re.match(r"run_(\d+)$", path_obj.name)
    return int(m.group(1)) if m else -1


def latest_run(run_root):
    root = Path(run_root)
    runs = [p for p in root.glob("run_*") if p.is_dir()]
    if not runs:
        return None
    runs.sort(key=run_index)
    return str(runs[-1])


def best_checkpoint(run_dir):
    if run_dir is None:
        return None
    for name in ["generator_best.pth", "generator_final.pth"]:
        p = Path(run_dir) / name
        if p.exists():
            return str(p)
    return None


def metrics_csv(run_dir):
    if run_dir is None:
        return None
    for name in ["metrics_test.csv", "metrics_val.csv"]:
        p = Path(run_dir) / name
        if p.exists():
            return str(p)
    return None


BASE_RUN_ROOT = os.path.join(OUTPUT_ROOT, "base")
EXPR_RUN_ROOT = os.path.join(OUTPUT_ROOT, "expr")

base_run_dir = latest_run(BASE_RUN_ROOT)
expr_run_dir = latest_run(EXPR_RUN_ROOT)

print("Latest base run:", base_run_dir)
print("Latest expr run:", expr_run_dir)


In [ ]:
# Step 3: Evaluate M_base
import subprocess

base_ckpt = best_checkpoint(base_run_dir)
assert base_ckpt is not None, f"No checkpoint found in {base_run_dir}"
print("Using base checkpoint:", base_ckpt)
subprocess.run(["python", "eval.py", "--config", "configs/base.colab.yaml", "--checkpoint", base_ckpt], check=True)


In [ ]:
# Step 4: Evaluate M_expr
import subprocess

expr_ckpt = best_checkpoint(expr_run_dir)
assert expr_ckpt is not None, f"No checkpoint found in {expr_run_dir}"
print("Using expr checkpoint:", expr_ckpt)
subprocess.run(["python", "eval.py", "--config", "configs/expr.colab.yaml", "--checkpoint", expr_ckpt], check=True)


In [ ]:
# Step 5: Generate comparison report
import subprocess

base_metrics_csv = metrics_csv(base_run_dir)
expr_metrics_csv = metrics_csv(expr_run_dir)
assert base_metrics_csv is not None, f"No metrics CSV found in {base_run_dir}"
assert expr_metrics_csv is not None, f"No metrics CSV found in {expr_run_dir}"

report_dir = os.path.join(OUTPUT_ROOT, "report")
print("Base metrics:", base_metrics_csv)
print("Expr metrics:", expr_metrics_csv)
print("Report dir:", report_dir)

subprocess.run([
    "python", "report.py",
    "--base", base_metrics_csv,
    "--expr", expr_metrics_csv,
    "--output", report_dir,
], check=True)


In [ ]:
# Preview report outputs
import os
import pandas as pd
from IPython.display import display, Image as IPImage

summary_csv = os.path.join(report_dir, "summary_table.csv")
hyp_csv = os.path.join(report_dir, "hypothesis_table.csv")

if os.path.exists(summary_csv):
    print("Summary table")
    display(pd.read_csv(summary_csv))

if os.path.exists(hyp_csv):
    print("Hypothesis table")
    display(pd.read_csv(hyp_csv))

for fig_name in ["metric_comparison.png", "expression_comparison.png", "structure_metrics.png"]:
    fig_path = os.path.join(report_dir, fig_name)
    if os.path.exists(fig_path):
        print("\n", fig_name)
        display(IPImage(filename=fig_path))


## Resume Training (Optional)

To resume, edit one of the generated Colab config files, set:
- `training.resume` to a saved `checkpoint_epoch_XXX.pth`

Then rerun the corresponding training cell.
